---
# Stage 5 – Random Forest Classifier

In [ ]:
# ─── Stage 5: Random Forest ───────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib, os, time, warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                              recall_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)

# ── Load arrays (saved by Stage 3) ───────────────────────────────────────────
X_train_vt  = np.load('preprocessed/X_train_vt.npy')
X_test_vt   = np.load('preprocessed/X_test_vt.npy')
y_train_enc = np.load('preprocessed/y_train_enc.npy')
y_test_enc  = np.load('preprocessed/y_test_enc.npy')

# ── Load UCI feature names for interpretable charts ──────────────────────────
# features.txt has 561 names; apply the same VarianceThreshold mask to align
try:
    _raw_names = pd.read_csv(
        'UCI HAR Dataset/features.txt',
        sep=r'\s+', header=None, index_col=0
    )[1].values
    _vt = joblib.load('variance_threshold.pkl')
    feature_names = _raw_names[_vt.get_support()]   # keep only selected cols
    print(f'Feature names loaded: {len(feature_names)} after VT filter')
except Exception as e:
    feature_names = np.array([f'feat_{i}' for i in range(X_train_vt.shape[1])])
    print(f'features.txt not found ({e}) — using generic labels')

print(f'X_train_vt  : {X_train_vt.shape}')
print(f'X_test_vt   : {X_test_vt.shape}')
print(f'y_train_enc : {y_train_enc.shape} | classes: {np.unique(y_train_enc)}')

In [ ]:
# ─── 5.1 Hyperparameter tuning — RandomizedSearchCV ──────────────────────────
param_dist = {
    'n_estimators':      [100, 200, 300, 500],
    'max_depth':         [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
    'max_features':      ['sqrt', 'log2']
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=20,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='accuracy',
    n_jobs=-1, verbose=1, random_state=42
)

t0 = time.time()
rf_search.fit(X_train_vt, y_train_enc)
print(f'Done in {(time.time()-t0)/60:.1f} min')
print(f'Best CV accuracy : {rf_search.best_score_*100:.2f}%')
print(f'Best params      : {rf_search.best_params_}')

In [ ]:
# ─── 5.2 Evaluate best Random Forest ─────────────────────────────────────────
best_rf   = rf_search.best_estimator_
y_pred_rf = best_rf.predict(X_test_vt)

rf_acc    = accuracy_score(y_test_enc, y_pred_rf)
rf_prec   = precision_score(y_test_enc, y_pred_rf, average='weighted')
rf_rec    = recall_score(y_test_enc, y_pred_rf, average='weighted')
rf_f1_mac = f1_score(y_test_enc, y_pred_rf, average='macro')
rf_f1_wt  = f1_score(y_test_enc, y_pred_rf, average='weighted')

print(f"{'='*55}")
print(f'  Random Forest (Tuned)')
print(f"{'='*55}")
print(f'  Accuracy          : {rf_acc*100:.2f}%')
print(f'  Precision (wt)    : {rf_prec:.4f}')
print(f'  Recall (wt)       : {rf_rec:.4f}')
print(f'  F1 Macro          : {rf_f1_mac:.4f}')
print(f'  F1 Weighted       : {rf_f1_wt:.4f}')
print(f'\n{classification_report(y_test_enc, y_pred_rf, target_names=CLASS_NAMES)}')

In [ ]:
# ─── 5.3 Confusion matrix ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay(
    confusion_matrix(y_test_enc, y_pred_rf),
    display_labels=CLASS_NAMES
).plot(ax=ax, colorbar=True, cmap='Blues', xticks_rotation=45)
ax.set_title('Confusion Matrix — Random Forest (Tuned)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('stage5_cm_RF.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: stage5_cm_RF.png')

In [ ]:
# ─── 5.4 Feature importance — mapped to real UCI feature names ────────────────
# FIX: use actual feature names from features.txt, not generic F40/F53 indices
importances = best_rf.feature_importances_
top_n   = 20
top_idx = np.argsort(importances)[::-1][:top_n]

imp_df = pd.DataFrame({
    'feature':    feature_names[top_idx],      # real UCI names (e.g. tBodyAcc-mean()-X)
    'importance': importances[top_idx]
})

fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(imp_df['feature'][::-1], imp_df['importance'][::-1],
        color='steelblue', alpha=0.85, edgecolor='none')
ax.axvline(importances.mean(), color='red', linestyle='--',
           alpha=0.6, label=f'Mean importance ({importances.mean():.4f})')
ax.set_xlabel('Mean Decrease in Impurity')
ax.set_title('Random Forest — Top 20 Feature Importances (UCI Names)', fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('stage5_rf_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: stage5_rf_feature_importance.png')
print('\nTop 10 features:')
print(imp_df.head(10).to_string(index=False))

In [ ]:
# ─── 5.5 Save model ───────────────────────────────────────────────────────────
os.makedirs('models', exist_ok=True)
joblib.dump(best_rf, 'models/random_forest_model.pkl')
print('Saved: models/random_forest_model.pkl')

---
# Stage 6 – LSTM Classifier

In [ ]:
# ─── Stage 6: LSTM — Load sequential data ────────────────────────────────────
import tensorflow as tf
tf.random.set_seed(42)
np.random.seed(42)

X_raw_train = np.load('preprocessed/X_raw_train_norm.npy')
X_raw_test  = np.load('preprocessed/X_raw_test_norm.npy')
y_train_ohe = np.load('preprocessed/y_train_ohe.npy')
y_test_ohe  = np.load('preprocessed/y_test_ohe.npy')

N_STEPS = X_raw_train.shape[1]   # 128 timesteps
N_FEATS = X_raw_train.shape[2]   # 9 sensor channels
N_CLS   = y_train_ohe.shape[1]   # 6 classes

print(f'X_raw_train : {X_raw_train.shape}  ({N_STEPS} timesteps x {N_FEATS} channels)')
print(f'X_raw_test  : {X_raw_test.shape}')
print(f'y_train_ohe : {y_train_ohe.shape}  ({N_CLS} classes)')

In [ ]:
# ─── 6.1 Compute class weights to handle Sitting/Standing confusion ───────────
# Sitting and Standing are static postures with very similar sensor readings.
# Class weights penalise the model more for misclassifying minority classes,
# which helps reduce the Sitting→Standing confusion seen in baseline runs.
from sklearn.utils.class_weight import compute_class_weight

y_train_int = np.argmax(y_train_ohe, axis=1)   # integer labels for weight calc
class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(N_CLS),
    y=y_train_int
)
class_weight_dict = {i: w for i, w in enumerate(class_weights_arr)}

print('Class weights (balanced):')
for i, name in enumerate(CLASS_NAMES):
    print(f'  {i} {name:<22} : {class_weight_dict[i]:.4f}')

In [ ]:
# ─── 6.2 Build LSTM architecture ─────────────────────────────────────────────
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam

lstm_model = Sequential([
    LSTM(128, input_shape=(N_STEPS, N_FEATS), return_sequences=True),
    BatchNormalization(),
    Dropout(0.3),
    LSTM(64, return_sequences=False),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(N_CLS, activation='softmax')
], name='HAR_LSTM')

lstm_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
lstm_model.summary()

In [ ]:
# ─── 6.3 Train with early stopping + class weights ───────────────────────────
# class_weight passed here directly addresses Sitting/Standing confusion
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=10,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=5, min_lr=1e-6, verbose=1)
]

history = lstm_model.fit(
    X_raw_train, y_train_ohe,
    epochs=100,
    batch_size=64,
    validation_split=0.1,
    class_weight=class_weight_dict,     # <── key fix for Sitting/Standing
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# ─── 6.4 Training curves ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history.history['accuracy'],     label='Train')
axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history.history['loss'],     label='Train')
axes[1].plot(history.history['val_loss'], label='Val')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('LSTM Training Curves', fontweight='bold')
plt.tight_layout()
plt.savefig('stage6_lstm_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: stage6_lstm_training_curves.png')

In [ ]:
# ─── 6.5 Evaluate LSTM ────────────────────────────────────────────────────────
y_prob_lstm = lstm_model.predict(X_raw_test, verbose=0)
y_pred_lstm = np.argmax(y_prob_lstm, axis=1)
y_true_lstm = np.argmax(y_test_ohe,  axis=1)

lstm_acc    = accuracy_score(y_true_lstm, y_pred_lstm)
lstm_prec   = precision_score(y_true_lstm, y_pred_lstm, average='weighted')
lstm_rec    = recall_score(y_true_lstm, y_pred_lstm, average='weighted')
lstm_f1_mac = f1_score(y_true_lstm, y_pred_lstm, average='macro')
lstm_f1_wt  = f1_score(y_true_lstm, y_pred_lstm, average='weighted')

print(f"{'='*55}")
print(f'  LSTM (Stacked, class-weighted)')
print(f"{'='*55}")
print(f'  Accuracy          : {lstm_acc*100:.2f}%')
print(f'  Precision (wt)    : {lstm_prec:.4f}')
print(f'  Recall (wt)       : {lstm_rec:.4f}')
print(f'  F1 Macro          : {lstm_f1_mac:.4f}')
print(f'  F1 Weighted       : {lstm_f1_wt:.4f}')
print(f'\n{classification_report(y_true_lstm, y_pred_lstm, target_names=CLASS_NAMES)}')

In [ ]:
# ─── 6.6 Confusion matrix ─────────────────────────────────────────────────────
cm_lstm = confusion_matrix(y_true_lstm, y_pred_lstm)

fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay(cm_lstm, display_labels=CLASS_NAMES).plot(
    ax=ax, colorbar=True, cmap='Purples', xticks_rotation=45)
ax.set_title('Confusion Matrix — LSTM (with class weights)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('stage6_cm_LSTM.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: stage6_cm_LSTM.png')

In [ ]:
# ─── 6.7 Sitting / Standing confusion analysis ────────────────────────────────
# Sitting (class 3) and Standing (class 4) are static postures.
# The phone is held still in both — no periodic motion to distinguish them.
# Only subtle gravity vector orientation differs, making them hard for any model.

sit_idx   = CLASS_NAMES.index('Sitting')    # 3
stand_idx = CLASS_NAMES.index('Standing')   # 4

# Count how many Sitting windows were called Standing and vice versa
sit_as_stand   = cm_lstm[sit_idx,   stand_idx]
stand_as_sit   = cm_lstm[stand_idx, sit_idx]
total_sit      = cm_lstm[sit_idx,   :].sum()
total_stand    = cm_lstm[stand_idx, :].sum()

print('=== Sitting / Standing Confusion Analysis ===')
print(f'  Sitting misclassified as Standing : {sit_as_stand} / {total_sit}'
      f'  ({sit_as_stand/total_sit*100:.1f}%)')
print(f'  Standing misclassified as Sitting : {stand_as_sit} / {total_stand}'
      f'  ({stand_as_sit/total_stand*100:.1f}%)')
print()
print('Why this happens:')
print('  Both activities are static — no limb movement, near-zero acceleration.')
print('  The only difference is the gravity vector angle (hip tilt vs upright).')
print('  Raw 128-step windows carry very similar signal patterns for both.')
print()
print('Fixes applied in this notebook:')
print('  1. class_weight="balanced" added to lstm_model.fit() — increases')
print('     penalty for misclassifying under-represented windows.')
print()
print('Further improvements (beyond this notebook):')
print('  - Add gravity-angle features (arctan of mean acc channels)')
print('  - Use longer context windows (256 steps) to capture posture stability')
print('  - Focal loss instead of cross-entropy for hard-to-separate classes')

# ── Zoom-in plot: just Sitting vs Standing cell ───────────────────────────────
zoom = cm_lstm[sit_idx:stand_idx+1, sit_idx:stand_idx+1]
fig, ax = plt.subplots(figsize=(4, 3))
ConfusionMatrixDisplay(zoom, display_labels=['Sitting', 'Standing']).plot(
    ax=ax, colorbar=False, cmap='Reds')
ax.set_title('Sitting vs Standing (zoom)', fontweight='bold')
plt.tight_layout()
plt.savefig('stage6_sitting_standing_zoom.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: stage6_sitting_standing_zoom.png')

In [ ]:
# ─── 6.8 Save LSTM model ──────────────────────────────────────────────────────
lstm_model.save('models/lstm_model.h5')
print('Saved: models/lstm_model.h5')

---
# Stage 7 – Model Comparison

In [ ]:
# ─── Final comparison: SVM (teammate) | RF | LSTM ────────────────────────────
comparison = pd.DataFrame([
    {'Model': 'Linear SVM',    'Accuracy (%)': 96.47, 'F1 Macro': 0.9650, 'F1 Weighted': 0.9645},
    {'Model': 'Random Forest', 'Accuracy (%)': round(rf_acc*100,    2),
     'F1 Macro': round(rf_f1_mac,  4), 'F1 Weighted': round(rf_f1_wt,  4)},
    {'Model': 'LSTM',          'Accuracy (%)': round(lstm_acc*100,  2),
     'F1 Macro': round(lstm_f1_mac,4), 'F1 Weighted': round(lstm_f1_wt, 4)},
])

print('=== Final Model Comparison ===')
display(comparison.style.highlight_max(
    subset=['Accuracy (%)', 'F1 Macro', 'F1 Weighted'], color='lightgreen'
))

In [ ]:
# ─── Comparison bar chart ─────────────────────────────────────────────────────
models   = comparison['Model'].tolist()
acc_vals = comparison['Accuracy (%)'].tolist()
f1m_vals = (comparison['F1 Macro']    * 100).tolist()
f1w_vals = (comparison['F1 Weighted'] * 100).tolist()

x, w = np.arange(len(models)), 0.25
fig, ax = plt.subplots(figsize=(10, 5))

b1 = ax.bar(x - w, acc_vals, w, label='Accuracy (%)',    color='steelblue', alpha=0.85)
b2 = ax.bar(x,     f1m_vals, w, label='F1 Macro (%)',    color='tomato',    alpha=0.85)
b3 = ax.bar(x + w, f1w_vals, w, label='F1 Weighted (%)', color='seagreen',  alpha=0.85)

for bars in [b1, b2, b3]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=11)
ax.set_ylim(80, 100)
ax.set_ylabel('Score (%)')
ax.set_title('Model Comparison — UCI HAR', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('stage7_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: stage7_model_comparison.png')

In [ ]:
# ─── Download all outputs (Colab only) ────────────────────────────────────────
from google.colab import files

for f in [
    'models/random_forest_model.pkl',
    'models/lstm_model.h5',
    'stage5_cm_RF.png',
    'stage5_rf_feature_importance.png',
    'stage6_cm_LSTM.png',
    'stage6_lstm_training_curves.png',
    'stage6_sitting_standing_zoom.png',
    'stage7_model_comparison.png'
]:
    files.download(f)
    print(f'Downloaded: {f}')